# AI-Task5: Dynamic Prompt Engine (LangChain)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaweriamushtaq41-coder/Smart_District_2/blob/main/AI-Task5/AI_Task5_Dynamic_Prompt_Engine.ipynb)

**Kya banana hai:** Topic aur audience level se `ChatPromptTemplate` banana, do alag providers ko LangChain wrappers se bhejna aur dono jawab live stream karna (provider ke label ke saath).

**Concepts:** LangChain, ChatPromptTemplate, messages, `stream()`

**Colab mein kaise chalayen:**
1. Upar *Open In Colab* dabao (ya ye file Colab mein upload karo).
2. *Runtime > Run all* dabao.
3. Keys ho to Colab ke left side ke **Secrets** (chaabi ka icon) mein daalo. Bina key ke bhi chalta hai (**demo mode**, nakli jawab).

> Notebook ko share karne se pehle apni keys kabhi output / cell mein mat chhodo.

## 1. Libraries install aur keys (`.env`)

In [1]:
!pip -q install -U python-dotenv langchain-core langchain-openai langchain-anthropic langchain-google-genai langchain-groq

In [2]:
import os

# .env file banao. Keys do tareeqon se aa sakti hain:
#  1) Colab "Secrets" (left side ka chaabi wala icon), secret ka naam wahi jo neeche list mein hai, "Notebook access" ON
#  2) apni .env file Colab ke "Files" panel (left side ka folder icon) mein drag-drop kar do, run karne se pehle
NAMES = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY"]

existing = {}                       # pehle se .env ho (upload ki hui) to uski keys bacha lo
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if "=" in line and not line.startswith("#"):
                key, value = line.strip().split("=", 1)
                existing[key] = value

lines = []
for name in NAMES:
    value = existing.get(name, "")
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:          # Colab nahi hai, ya ye secret nahi banaya
        pass
    lines.append(f"{name}={value}")

with open(".env", "w") as f:
    f.write("\n".join(lines) + "\n")

from dotenv import load_dotenv
load_dotenv(".env", override=True)

found = [n for n in NAMES if os.getenv(n)]
print("Keys mili:", found if found else "koi nahi (demo mode chalega)")

Keys mili: koi nahi (demo mode chalega)


## 2. Model factory
`get_chat_model("groq")` jaisa call asli LangChain model deta hai (ChatOpenAI, ChatAnthropic, ChatGoogleGenerativeAI, ChatGroq).
Provider badalna ho to sirf naam badlo, chain wahi rehti hai. `demo_model` / `demo_stream_model` key ke baghair nakli model dete hain jo chain mein asli ki tarah lagte hain.

In [3]:
import os

from dotenv import load_dotenv
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

load_dotenv()

PROVIDERS = ["openai", "claude", "gemini", "groq"]

KEY_NAMES = {
    "openai": "OPENAI_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
}

MODELS = {
    "openai": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    "claude": os.getenv("ANTHROPIC_MODEL", "claude-3-5-haiku-latest"),
    "gemini": os.getenv("GEMINI_MODEL", "gemini-1.5-flash"),
    "groq": os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"),
}


def available_providers():
    """Jin providers ki key .env mein hai."""
    return [p for p in PROVIDERS if os.getenv(KEY_NAMES[p], "").strip()]


def get_chat_model(provider, temperature=0.3):
    """Asli LangChain model wapas deta hai. Import andar hai taake sirf wahi package chahiye jo use ho."""
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODELS["openai"], temperature=temperature)
    if provider == "claude":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=MODELS["claude"], temperature=temperature)
    if provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=MODELS["gemini"], temperature=temperature)
    if provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=MODELS["groq"], temperature=temperature)
    raise ValueError(f"Unknown provider: {provider}")


# ---------------- demo (nakli) models ----------------
def demo_stream_model(text, delay=0.004):
    """Ek fixed jawab deta hai, harf harf stream hota hai (asli streaming jaisa)."""
    return FakeListChatModel(responses=[text], sleep=delay)


def demo_model(responder):
    """responder(prompt_text) -> jawab ka text. Prompt dekh kar jawab chunta hai,
    is liye RunnableParallel mein (jahan tarteeb pakki nahi) bhi theek chalta hai."""
    def run(prompt_value):
        text = prompt_value if isinstance(prompt_value, str) else prompt_value.to_string()
        return AIMessage(content=responder(text))
    return RunnableLambda(run)

## 3. Prompt template aur streaming
- `LEVEL_STYLE`: audience ke hisab se andaaz
- `prompt`: system + human message ka template (`{level}`, `{style}`, `{topic}` baad mein bharte hain)
- `stream_answer`: `prompt | model | StrOutputParser()` chain ka `.stream()` jawab tukron mein print karta hai

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [5]:
LEVEL_STYLE = {
    "beginner": "Use very simple words, a daily-life example and no jargon. Max 4 sentences.",
    "intermediate": "Assume basic programming knowledge. Give a short code-free explanation and one practical use. Max 5 sentences.",
    "expert": "Be precise and technical. Mention trade-offs and edge cases. Max 5 sentences.",
}

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly teacher. Audience level: {level}. {style}"),
    ("human", "Explain this topic: {topic}"),
])

DEMO_TEXT = {
    "beginner": ("{topic} ko aise samjho jaise raaste ka naqsha: pehle chhota qadam, phir agla qadam. "
                 "Roz ki zindagi mein ye aisa hai jaise chai banane ka tareeqa, har step ke baad agla step. "
                 "Is liye {topic} seekhna aasan hai."),
    "intermediate": ("{topic} ek aisa concept hai jo bare masle ko chhote hisson mein baant deta hai. "
                     "Practical kaam mein ye code ko saaf aur dobara istemal ke qabil banata hai."),
    "expert": ("{topic}: asal faida modularity aur predictable behaviour hai, magar trade-off performance "
               "aur debugging ki sakhti mein hota hai. Edge cases (khali input, bare data) pehle sochne chahiye."),
}


# demo mein dono jawab thore alag dikhane ke liye
DEMO_TAIL = {"openai": " (OpenAI andaaz: tafseeli)", "groq": " (Groq andaaz: mukhtasar aur tez)",
             "claude": " (Claude andaaz: narm lehja)", "gemini": " (Gemini andaaz: misalon wala)"}


def pick_providers(arg, demo):
    if arg:
        chosen = [p.strip().lower() for p in arg.split(",")]
    elif demo:
        chosen = ["openai", "groq"]
    else:
        chosen = available_providers()[:2]
    return chosen


def stream_answer(label, model, variables):
    chain = prompt | model | StrOutputParser()      # prompt -> model -> text
    print(f"\n[{label.upper()}] ", end="", flush=True)
    for chunk in chain.stream(variables):           # jawab tukron mein aata hai
        print(chunk, end="", flush=True)
    print()

## 4. Chalao
`TOPIC` aur `LEVEL` badlo. Do providers ki keys hon to wahi chalenge, warna demo.

In [6]:
TOPIC = "Recursion"
LEVEL = "beginner"            # beginner / intermediate / expert
CHOSEN = None                 # ya ["openai", "groq"]
FORCE_DEMO = False

demo = FORCE_DEMO
providers = pick_providers(",".join(CHOSEN) if CHOSEN else None, demo)
if not demo and len(providers) < 2:
    print("[!] Do providers ki keys nahi mili, DEMO mode chal raha hai.\n")
    demo, providers = True, ["openai", "groq"]

variables = {"topic": TOPIC, "level": LEVEL, "style": LEVEL_STYLE[LEVEL]}

print("--- Banaya gaya prompt ---")
for message in prompt.format_messages(**variables):          # SystemMessage, HumanMessage
    print(f"{type(message).__name__}: {message.content}")

for name in providers:
    if demo:
        model = demo_stream_model(DEMO_TEXT[LEVEL].format(topic=TOPIC) + DEMO_TAIL.get(name, ""))
    else:
        model = get_chat_model(name)
    stream_answer(name + (" (demo)" if demo else ""), model, variables)

[!] Do providers ki keys nahi mili, DEMO mode chal raha hai.

--- Banaya gaya prompt ---
SystemMessage: You are a friendly teacher. Audience level: beginner. Use very simple words, a daily-life example and no jargon. Max 4 sentences.
HumanMessage: Explain this topic: Recursion

[OPENAI (DEMO)] Recursion ko aise samjho jaise raaste ka naqsha: pehle chhota qadam, phir agla qadam. Roz ki zindagi mein ye aisa hai jaise chai banane ka tareeqa, har step ke baad agla step. Is liye Recursion seekhna aasan hai. (OpenAI andaaz: tafseeli)

[GROQ (DEMO)] Recursion ko aise samjho jaise raaste ka naqsha: pehle chhota qadam, phir agla qadam. Roz ki zindagi mein ye aisa hai jaise chai banane ka tareeqa, har step ke baad agla step. Is liye Recursion seekhna aasan hai. (Groq andaaz: mukhtasar aur tez)


## Kya seekha
- `ChatPromptTemplate` mein variables baad mein bharte hain, is liye ek template kai audiences ke liye chalta hai.
- `invoke` ek jawab, `stream` tukron mein, `batch` kai inputs ek saath.
- LangChain wrapper ki wajah se OpenAI / Claude / Gemini / Groq ka code ek jaisa hai.